# Ch08_02_Jena_Climate

『실전! AI 프로젝트 완주의 기술』의 인쇄용 PDF에 수록된 코드에서 추출한 학습용 노트북입니다.

**주의:** 인쇄물에서 자동 추출된 초안입니다. 일부 조판 문자·줄바꿈·별도 데이터 경로가 원본 소스와 다를 수 있으므로 실행 전 확인하세요. 각 코드 셀 위에 인쇄본 페이지를 표시했습니다. 데이터 다운로드 및 패키지 설치는 필요한 경우 직접 진행해야 합니다. 출판 전 실행 검증본으로 간주하지 마세요.

**인쇄본 215쪽 — 코드**

In [1]:
import pandas as pd
data = pd.read_csv('jena_climate_2009_2016.csv')
print(data.shape)

(420551, 15)


**인쇄본 215쪽 — 출력 예시 (실행 코드 아님)**

```text
(420551, 15)
```

**인쇄본 215쪽 — 코드**

In [2]:
print(data[['Date Time', 'T (degC)', 'rh (%)']].head())

             Date Time  T (degC)  rh (%)
0  01.01.2009 00:10:00     -8.02    93.3
1  01.01.2009 00:20:00     -8.41    93.4
2  01.01.2009 00:30:00     -8.51    93.9
3  01.01.2009 00:40:00     -8.31    94.2
4  01.01.2009 00:50:00     -8.27    94.1


**인쇄본 215쪽 — 출력 예시 (실행 코드 아님)**

```text
             Date Time  T (degC)  rh (%)
0  01.01.2009 00:10:00     -8.02    93.3
1  01.01.2009 00:20:00     -8.41    93.4
2  01.01.2009 00:30:00     -8.51    93.9
3  01.01.2009 00:40:00     -8.31    94.2
4  01.01.2009 00:50:00     -8.27    94.1
```

**인쇄본 215쪽 — 코드**

In [3]:
print((data['wv (m/s)'] == -9999.0).sum())
print((data['max. wv (m/s)'] == -9999.0).sum())

18
20


**인쇄본 216쪽 — 출력 예시 (실행 코드 아님)**

```text
18
20
```

**인쇄본 216쪽 — 코드**

In [4]:
for col in ['wv (m/s)', 'max. wv (m/s)']:
    missing = data[col] == -9999.0
    mean_value = data.loc[~missing, col].mean()
    data.loc[missing, col] = mean_value

**인쇄본 216쪽 — 코드**

In [5]:
data = data.drop(columns=['Date Time'])
print(data.shape)

(420551, 14)


**인쇄본 216쪽 — 출력 예시 (실행 코드 아님)**

```text
(420551, 14)
```

**인쇄본 216쪽 — 코드**

In [6]:
data = data.iloc[::6].copy()
print(data.shape)

(70092, 14)


**인쇄본 216쪽 — 출력 예시 (실행 코드 아님)**

```text
(70092, 14)
```

**인쇄본 217쪽 — 코드**

In [7]:
features = data.to_numpy()
labels = data[['T (degC)']].to_numpy()
print(features.shape)
print(labels.shape)

(70092, 14)
(70092, 1)


**인쇄본 217쪽 — 출력 예시 (실행 코드 아님)**

```text
(70092, 14)
(70092, 1)
```

**인쇄본 217쪽 — 코드** — 

In [8]:
from sklearn.model_selection import train_test_split
X_rest, X_test, y_rest, y_test = train_test_split(
    features, labels,
    test_size=0.1,
    shuffle=False
)
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest,
    test_size=2/9,
    shuffle=False
)
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(49063, 14)
(14019, 14)
(7010, 14)


**인쇄본 218쪽 — 출력 예시 (실행 코드 아님)**

```text
(49063, 14)
(14019, 14)
(7010, 14)
```

**인쇄본 218쪽 — 코드**

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

**인쇄본 219쪽 — 코드**

In [10]:
from tensorflow.keras.preprocessing import (
    timeseries_dataset_from_array
)
T = 24
batch_size = 16
dataset_train = timeseries_dataset_from_array(
    X_train,
    y_train[T:],
    sequence_length=T,
    sequence_stride=1,
    batch_size=batch_size,
    shuffle=True
)

**인쇄본 220쪽 — 코드**

In [11]:
dataset_val = timeseries_dataset_from_array(
    X_val,
    y_val[T:],
    sequence_length=T,
    sequence_stride=1,
    batch_size=batch_size,
    shuffle=False
)
dataset_test = timeseries_dataset_from_array(
    X_test,
    y_test[T:],
    sequence_length=T,
    sequence_stride=1,
    batch_size=batch_size,
    shuffle=False
)

**인쇄본 220쪽 — 코드**

In [12]:
for inputs, targets in dataset_train.take(1):
    print(inputs.shape)
    print(targets.shape)

(16, 24, 14)
(16, 1)


**인쇄본 220쪽 — 출력 예시 (실행 코드 아님)**

```text
(16, 24, 14)
(16, 1)
```

**인쇄본 221쪽 — 코드**

In [13]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

**인쇄본 222쪽 — 코드**

In [14]:
model = Sequential([
    Input(shape=(T, 14)),
    LSTM(32),
    Dense(1)
])

**인쇄본 222쪽 — 코드**

In [15]:
from tensorflow.keras.metrics import RootMeanSquaredError
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=[RootMeanSquaredError()]
)

**인쇄본 222쪽 — 코드** — 

In [16]:
history = model.fit(
    dataset_train,
    epochs=30,
    validation_data=dataset_val,
    verbose=0
)

**인쇄본 223쪽 — 코드**

In [17]:
test_loss, test_rmse = model.evaluate(
    dataset_test,
    verbose=0
)
test_nrmse = test_rmse / y_test[T:].std()
print("RMSE =", test_rmse)
print("NRMSE =", test_nrmse)

RMSE = 0.6917944550514221
NRMSE = 0.08825668445448752


**인쇄본 223쪽 — 코드**

In [18]:
RMSE = 0.6962
NRMSE = 0.0888